In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent

# =========================
# Paths
# =========================

DATA_RAW = ROOT / "data" / "raw"
DATA_EXTERNAL = ROOT / "data" / "external" / "cresci2017"
DATA_PROCESSED = ROOT / "data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# =========================
# Load Dataset A
# =========================

train_df = pd.read_csv(
    DATA_RAW / "instagram_fake_spammer" / "train.csv"
)

test_df = pd.read_csv(
    DATA_RAW / "instagram_fake_spammer" / "test.csv"
)

instagram_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

# =========================
# Load InstaFake
# =========================

fake_df = pd.read_json(
    DATA_RAW / "instafake" / "fake-v1.0" / "fakeAccountData.json"
)

real_df = pd.read_json(
    DATA_RAW / "instafake" / "fake-v1.0" / "realAccountData.json"
)

fake_df["fake"] = 1
real_df["fake"] = 0

instafake_df = pd.concat(
    [fake_df, real_df],
    ignore_index=True
)

# =========================
# Load Cresci
# =========================

genuine_df = pd.read_csv(
    DATA_EXTERNAL / "genuine_users.csv"
)

spam_df = pd.read_csv(
    DATA_EXTERNAL / "spam_user.csv"
)

genuine_df["fake"] = 0
spam_df["fake"] = 1

cresci_df = pd.concat(
    [genuine_df, spam_df],
    ignore_index=True
)

print("Instagram:", instagram_df.shape)
print("InstaFake:", instafake_df.shape)
print("Cresci:", cresci_df.shape)

Instagram: (696, 12)
InstaFake: (1194, 10)
Cresci: (4465, 43)


In [2]:
# Rename Dataset A columns to unified names

instagram_df = instagram_df.rename(columns={
    "profile pic": "has_profile_picture",
    "description length": "bio_length",
    "#posts": "posts",
    "#followers": "followers",
    "#follows": "following",
    "private": "is_private",
    "nums/length username": "username_digit_ratio"
})

# Select useful common features
instagram_clean = instagram_df[
    [
        "followers",
        "following",
        "posts",
        "bio_length",
        "has_profile_picture",
        "is_private",
        "username_digit_ratio",
        "fake"
    ]
].copy()

print(instagram_clean.head())
print(instagram_clean.shape)

   followers  following  posts  bio_length  has_profile_picture  is_private  \
0       1000        955     32          53                    1           0   
1       2740        533    286          44                    1           0   
2        159         98     13           0                    1           1   
3        414        651    679          82                    1           0   
4        151        126      6           0                    1           1   

   username_digit_ratio  fake  
0                  0.27     0  
1                  0.00     0  
2                  0.10     0  
3                  0.00     0  
4                  0.00     0  
(696, 8)


In [3]:
instafake_clean = instafake_df.rename(columns={
    "userFollowerCount": "followers",
    "userFollowingCount": "following",
    "userBiographyLength": "bio_length",
    "userMediaCount": "posts",
    "userHasProfilPic": "has_profile_picture",
    "userIsPrivate": "is_private"
})

# Calculate username digit ratio
instafake_clean["username_digit_ratio"] = (
    instafake_clean["usernameDigitCount"]
    / instafake_clean["usernameLength"].replace(0, np.nan)
)

# Keep common features
instafake_clean = instafake_clean[
    [
        "followers",
        "following",
        "posts",
        "bio_length",
        "has_profile_picture",
        "is_private",
        "username_digit_ratio",
        "fake"
    ]
].copy()

print(instafake_clean.head())
print(instafake_clean.shape)

   followers  following  posts  bio_length  has_profile_picture  is_private  \
0         25       1937      0           0                    1           1   
1        324       4122      0           0                    1           0   
2         15        399      0           0                    0           0   
3         14        107      1           0                    1           0   
4        264       4651      0           0                    1           0   

   username_digit_ratio  fake  
0              0.000000     1  
1              0.266667     1  
2              0.250000     1  
3              0.100000     1  
4              0.000000     1  
(1194, 8)


In [4]:
cresci_clean = cresci_df.copy()

cresci_clean = cresci_clean.rename(columns={
    "followers_count": "followers",
    "friends_count": "following",
    "statuses_count": "posts",
    "protected": "is_private"
})

# Bio length
cresci_clean["bio_length"] = (
    cresci_clean["description"]
    .fillna("")
    .astype(str)
    .str.len()
)

# Username digit ratio
username = cresci_clean["screen_name"].fillna("").astype(str)

cresci_clean["username_digit_ratio"] = (
    username.str.count(r"\d")
    / username.str.len().replace(0, np.nan)
)

# Profile picture
# default_profile_image = 1 generally indicates default image
cresci_clean["has_profile_picture"] = (
    1 - cresci_clean["default_profile_image"].fillna(1)
)

# Select common features
cresci_clean = cresci_clean[
    [
        "followers",
        "following",
        "posts",
        "bio_length",
        "has_profile_picture",
        "is_private",
        "username_digit_ratio",
        "fake"
    ]
].copy()

print(cresci_clean.head())
print(cresci_clean.shape)

   followers  following  following   posts  bio_length  has_profile_picture  \
0        208        332        NaN    2177          21                  0.0   
1        330        485        NaN    2660          48                  0.0   
2        166        177        NaN    1254          34                  0.0   
3       2248        981        NaN  202968          58                  0.0   
4         21         79        NaN      82          13                  0.0   

   is_private  username_digit_ratio  fake  
0         NaN              0.500000     0  
1         NaN              0.500000     0  
2         NaN              0.222222     0  
3         NaN              0.000000     0  
4         NaN              0.533333     0  
(4465, 9)


In [5]:
instagram_clean.to_csv(
    DATA_PROCESSED / "instagram_processed.csv",
    index=False
)

instafake_clean.to_csv(
    DATA_PROCESSED / "instafake_processed.csv",
    index=False
)

cresci_clean.to_csv(
    DATA_PROCESSED / "cresci2017_processed.csv",
    index=False
)

print("✅ Processed datasets saved.")

✅ Processed datasets saved.
